### Maximal Marginal Relevance
MMR (Maximal Marginal Relevance) is a powerful diversity-aware retrieval technique used in information retrieval and RAG pipelines to balance relevance and novelty when selecting documents.

In [22]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project Root:", PROJECT_ROOT)

Project Root: d:\Practice\AI\RAG-Bootcamp


In [23]:
from imports import *

load_dotenv()

True

In [24]:

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")


In [25]:
# Step 1: Load and chunk the document
loader = TextLoader("langchain_rag_dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)
chunks

[Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs).\nLangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='The framework supports integration with various vector databases like FAISS and Chroma for semantic retrieval.\nLangChain enables Retrieval-Augmented Generation (RAG) by allowing developers to fetch relevant context before generating responses.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
 Document(metadata={'

In [26]:
# Step 2: FAISS Vector Store with HuggingFace Embeddings
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding_model)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2886.67it/s]


In [27]:
### Step 3: Create MMR Retirever
retriever=vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k":3}
)

In [28]:
# Step 4: Prompt and LLM
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question:
{question}
""")
llm=init_chat_model("openai:gpt-3.5-turbo")


In [29]:
# Step 5: RAG Pipeline
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [30]:
# Step 6: Query
from langchain_core.runnables import RunnableMap
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    RunnableMap(
    {
        "context": lambda x: format_docs(retriever.invoke(x["question"])),
        "question": lambda x: x["question"],
    }
)
    | prompt
    | llm
    | StrOutputParser()
)

In [31]:
response = rag_chain.invoke(
    {
        "question": "What is LangChain?"
    }
)

print(response)

LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs). It provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems. LangChain also supports hybrid retrieval strategies by combining BM25 and vector-based retrieval, and leverages the high-performance FAISS library for efficient retrieval in RAG pipelines. Additionally, LangChain often uses Chroma as a lightweight vector store for embedding-based document storage and retrieval, and supports prompt templates with Jinja-style formatting for customizing model inputs.
